# Recess Ready: Philadelphia

**Question:** each evening, can we predict whether tomorrow will be unsafe for outdoor recess at Philadelphia schools, because of **heat** or **polluted air**?

*Prototype, not official guidance.*

## Step 1: Download Philadelphia's daily weather and air quality

We ask Open-Meteo (free, no API key) for one row per day in Philadelphia:
- **Weather** (archive API): the hottest "feels like" temperature of the day.
- **Air quality** (air-quality API): PM2.5 (tiny particles) every hour. We average the hours into one number per day.

Note: Open-Meteo's air quality numbers are **estimates from a computer model (CAMS)**, not readings from a sensor. We say this in our write-up.

In [ ]:
import pandas as pd
import requests

LAT, LON = 39.95, -75.16            # Philadelphia (Center City)
START, END = "2022-08-01", "2026-11-12"

# Weather: one row per day
weather_url = "https://archive-api.open-meteo.com/v1/archive"
weather = requests.get(weather_url, params={
    "latitude": LAT, "longitude": LON,
    "start_date": START, "end_date": END,
    "daily": "apparent_temperature_max",
    "timezone": "America/New_York",
}).json()
weather_df = pd.DataFrame(weather["daily"])
weather_df["time"] = pd.to_datetime(weather_df["time"])

# Air quality: one row per hour, then averaged per day
air_url = "https://air-quality-api.open-meteo.com/v1/air-quality"
air = requests.get(air_url, params={
    "latitude": LAT, "longitude": LON,
    "start_date": START, "end_date": END,
    "hourly": "pm2_5",
    "timezone": "America/New_York",
}).json()
air_df = pd.DataFrame(air["hourly"])
air_df["time"] = pd.to_datetime(air_df["time"])
air_daily = air_df.resample("D", on="time")["pm2_5"].mean().reset_index()

# Put weather and air side by side, one row per day
df = weather_df.merge(air_daily, on="time")
print(df.shape)
df.head()

## Step 2: How many unsafe *school* days are there?

We label a day **unsafe** if either:
- **Bad air:** daily average PM2.5 above **35.4 µg/m³**. That's the US EPA's "Unhealthy for Sensitive Groups" level, and kids count as a sensitive group.
- **Too hot:** the "feels like" temperature reaches **32°C (about 90°F)**. That's the US National Weather Service's "extreme caution" heat level.

We only count **school days**: weekdays from September to June. We skip July and August because there's no recess in summer. We ignore holidays for now.

**Why this check comes first:** if there are almost no unsafe school days, there's nothing for a model to learn. We'd change the thresholds or the city now, not on Sunday.

In [ ]:
df["bad_air"] = df["pm2_5"] > 35.4
df["too_hot"] = df["apparent_temperature_max"] >= 32
df["unsafe"] = df["bad_air"] | df["too_hot"]

is_weekday = df["time"].dt.weekday < 5           # Monday=0 ... Friday=4
is_school_month = ~df["time"].dt.month.isin([7, 8])
school = df[is_weekday & is_school_month].copy()
school["year"] = school["time"].dt.year

print("School days:", len(school))
print("Unsafe school days:", school["unsafe"].sum())
print(school.groupby("year")[["bad_air", "too_hot", "unsafe"]].sum())